In [5]:
import py_trees
import time  # 用于模拟延时

# 自定义条件节点：检查物体是否检测到（模拟基于 KF 的信念状态）
class CheckObjectDetected(py_trees.behaviour.Behaviour):
    def __init__(self, name):
        super(CheckObjectDetected, self).__init__(name)
        # 模拟 KF 的协方差 P（在实际中，从 KF 获取）
        self.blackboard = py_trees.blackboard.Blackboard()
        self.blackboard.set("kf_covariance_P", 0.05)  # 初始模拟值（小 P 表示高置信）

    def update(self):
        # 模拟检查：如果 P < 0.1（置信高），返回 SUCCESS
        P = self.blackboard.get("kf_covariance_P")
        if P < 0.1:
            self.feedback_message = f"Object detected (P={P} < 0.1)"
            return py_trees.common.Status.SUCCESS
        else:
            self.feedback_message = f"Detection uncertain (P={P} >= 0.1)"
            return py_trees.common.Status.FAILURE

# 自定义动作节点：模拟移动机械臂
class MoveArm(py_trees.behaviour.Behaviour):
    def __init__(self, name):
        super(MoveArm, self).__init__(name)
        self.counter = 0  # 模拟进度

    def update(self):
        self.counter += 1
        if self.counter < 3:  # 模拟运行中（需 3 次 Tick）
            self.feedback_message = f"Moving arm... (progress: {self.counter}/3)"
            return py_trees.common.Status.RUNNING
        else:
            self.feedback_message = "Arm moved successfully"
            return py_trees.common.Status.SUCCESS

# 构建 BT
def create_tree():
    # 根节点：Sequence（先检查检测，然后移动）
    root = py_trees.composites.Sequence(name="Root Sequence", memory=False)  # memory=False 表示不记住子节点状态

    # 添加子节点
    check_detect = CheckObjectDetected(name="Check Object Detected")
    move_arm = MoveArm(name="Move Arm")

    root.add_children([check_detect, move_arm])

    # 创建树
    tree = py_trees.trees.BehaviourTree(root)
    return tree

# 主函数：模拟运行
if __name__ == "__main__":
    tree = create_tree()

    # 打印树结构（调试用）
    print(py_trees.display.unicode_tree(tree.root))

    # 模拟 5 次 Tick（在实际机器人中放入循环）
    for i in range(5):
        print(f"\n--- Tick {i+1} ---")
        tree.tick()
        # 打印根节点状态
        print(f"Root Status: {tree.root.status}")
        # 打印反馈（可选）
        for node in [tree.root.children[0], tree.root.children[1]]:
            print(f"{node.name}: {node.feedback_message}")
        time.sleep(1)  # 模拟 Tick 间隔

    print("\nTask completed!")


[-] Root Sequence
    --> Check Object Detected
    --> Move Arm


--- Tick 1 ---
Root Status: Status.RUNNING
Check Object Detected: Object detected (P=0.05 < 0.1)
Move Arm: Moving arm... (progress: 1/3)

--- Tick 2 ---
Root Status: Status.RUNNING
Check Object Detected: Object detected (P=0.05 < 0.1)
Move Arm: Moving arm... (progress: 2/3)

--- Tick 3 ---
Root Status: Status.SUCCESS
Check Object Detected: Object detected (P=0.05 < 0.1)
Move Arm: Arm moved successfully

--- Tick 4 ---
Root Status: Status.SUCCESS
Check Object Detected: Object detected (P=0.05 < 0.1)
Move Arm: Arm moved successfully

--- Tick 5 ---
Root Status: Status.SUCCESS
Check Object Detected: Object detected (P=0.05 < 0.1)
Move Arm: Arm moved successfully

Task completed!
